<style>
.jp-RenderedHTMLCommon { font-size: 22px !important; line-height: 1.8 !important; }
.jp-RenderedHTMLCommon h1 { font-size: 2.2em !important; }
.jp-RenderedHTMLCommon h2 {
  font-size: 1.7em !important;
  border-left: 5px solid #f80e5e !important;
  padding: 10px 0 10px 18px !important;
  margin-top: 40px !important;
  margin-bottom: 16px !important;
  background: linear-gradient(90deg, rgba(248,14,94,0.08) 0%, transparent 60%) !important;
  border-radius: 4px !important;
}
.jp-RenderedHTMLCommon h3 {
  font-size: 1.35em !important;
  border-bottom: 2px solid rgba(248,14,94,0.3) !important;
  padding-bottom: 8px !important;
  margin-top: 30px !important;
  margin-bottom: 12px !important;
  color: #f80e5e !important;
}
.jp-RenderedHTMLCommon h4 {
  font-size: 1.15em !important;
  font-style: italic !important;
  color: #f80e5e !important;
  margin-top: 22px !important;
}
</style>

<div style="background: linear-gradient(135deg, #4A0E2E 0%, #880E4F 50%, #C2185B 100%); padding: 40px 45px; border-radius: 16px; margin-bottom: 30px; border: 1px solid rgba(248,14,94,0.15); max-width: 95%; overflow: hidden; box-sizing: border-box;">
  <div style="font-size: 15px; font-weight: 600; letter-spacing: 3px; text-transform: uppercase; color: #f80e5e; margin-bottom: 8px;">Chapter 09</div>
  <div style="font-size: 36px; font-weight: 800; color: #ffffff; margin-bottom: 16px; line-height: 1.2;">Text-to-Image Generation</div>
  <div style="width: 60px; height: 4px; background: linear-gradient(90deg, #f80e5e, rgba(248,14,94,0.5)); border-radius: 2px; margin-bottom: 18px;"></div>
  <div style="font-size: 17px; color: #b0bec5; line-height: 1.6;">Diffusion Models, U-Net, DiT & Classifier-Free Guidance</div>
  <div style="margin-top: 20px; font-size: 13px; color: #546e7a;">Source: ByteByteGo — Generative AI System Design Interview</div>
</div>

<div style="border-left: 5px solid #d90b5c; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(217,11,92,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Introduction</span>
</div>


In many cases, instead of allowing the model to generate content from random noise (as discussed in Chapters 7 and 8), we want to control the content of the generated image. Text-to-image generation is a fascinating application of generative AI that allows users to enter a text prompt, which the model transforms into a detailed image. Several text-to-image services are commercially available, such as OpenAI's DALL-E 3 [1], Google's Imagen [2], and Adobe's Firefly [3].


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-1-OXG3QF5W.svg" alt="Figure 1: An example of a prompt and a generated image by OpenAI’s DALL-E 3 [1]" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 1: An example of a prompt and a generated image by OpenAI’s DALL-E 3 [1]</p>
</div>



<div style="border-left: 5px solid #d90b5c; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(217,11,92,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Clarifying Requirements</span>
</div>


Here is a typical interaction between a candidate and an interviewer:


<div style="border: 1px solid rgba(248,14,94,0.3); border-radius: 8px; margin: 28px 0; overflow: hidden; font-family: system-ui, -apple-system, sans-serif;">
  <div style="background: rgba(248,14,94,0.08); padding: 10px 20px; border-bottom: 1px solid rgba(248,14,94,0.3); display: flex; align-items: center; gap: 8px;">
    <span style="font-size: 1.1em;">&#128483;</span>
    <span style="font-size: 0.85em; font-weight: 700; color: #f80e5e; text-transform: uppercase; letter-spacing: 1.5px;">Interview Transcript</span>
  </div>
  <div style="padding: 24px 20px 24px 28px; position: relative;">
    <div style="position: absolute; left: 38px; top: 24px; bottom: 24px; width: 2px; background: rgba(248,14,94,0.3);"></div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #f80e5e; border: 3px solid transparent; box-shadow: 0 0 0 2px #f80e5e; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #f80e5e; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #f80e5e; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">What resolution do we target for generated images?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">We aim for high-resolution images, specifically 1024x1024 pixels.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #f80e5e; border: 3px solid transparent; box-shadow: 0 0 0 2px #f80e5e; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #f80e5e; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #f80e5e; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Should the system support multiple languages for text input or just English?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">We'll focus on English initially, but the system's architecture should be adaptable for other languages later.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #f80e5e; border: 3px solid transparent; box-shadow: 0 0 0 2px #f80e5e; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #f80e5e; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #f80e5e; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">How large is the dataset for training a text-to-image model?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">We have about 500 million images from user assets, most with captions.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #f80e5e; border: 3px solid transparent; box-shadow: 0 0 0 2px #f80e5e; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #f80e5e; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #f80e5e; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">How detailed and complex can the text prompts be? Is there a limit to their complexity or length?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">The system should handle detailed text prompts, with a maximum length of 128 words.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #f80e5e; border: 3px solid transparent; box-shadow: 0 0 0 2px #f80e5e; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #f80e5e; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #f80e5e; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">What speed should the system achieve for image generation?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">The goal is near-real-time generation. Let’s aim for 10 seconds per image.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #f80e5e; border: 3px solid transparent; box-shadow: 0 0 0 2px #f80e5e; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #f80e5e; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #f80e5e; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">What types of images should the system generate? Are we focusing on a specific domain, like landscapes?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">The system should be capable of generating, based on text prompts, a wide range of images, including realistic landscapes, portraits, and abstract or conceptual art.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #f80e5e; border: 3px solid transparent; box-shadow: 0 0 0 2px #f80e5e; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #f80e5e; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #f80e5e; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">It's important to ensure the images aren't biased by age, race, or gender. Can I start by focusing on those three attributes?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Great point. It’s crucial to have a fair system. Let's begin by addressing those three attributes.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #f80e5e; border: 3px solid transparent; box-shadow: 0 0 0 2px #f80e5e; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #f80e5e; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #f80e5e; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Ethical considerations are crucial. We need filters and checks to avoid generating offensive, inappropriate, or harmful images. Does that sound correct?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 0; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Yes, that’s correct.</div>
      </div>
    </div>
  </div>
</div>


<div style="border-left: 5px solid #d90b5c; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(217,11,92,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Frame the Problem as an ML Task</span>
</div>


<div style="border-bottom: 2px solid rgba(248,14,94,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #f80e5e;">Specifying the system’s input and output</span>
</div>


The input to the system is a text prompt provided by the user that describes the desired image. This prompt usually includes details like scenes, objects, colors, styles, and emotions.

The output is a visually detailed image that adheres to the text prompt. For example, as shown in Figure 2, a prompt like "A boat on an ocean" produces an image depicting this scene.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-2-S5FPVU2H.svg" alt="Figure 2: Input and output of a text-to-image system. Image credit: [4]" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 2: Input and output of a text-to-image system. Image credit: [4]</p>
</div>



<div style="border-bottom: 2px solid rgba(248,14,94,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #f80e5e;">Choosing a suitable ML approach</span>
</div>


Text-to-image generation is a multimodal task that involves understanding text and generating a corresponding image. There are two primary approaches for building text-to-image systems:


* Autoregressive models
* Diffusion models


Let’s briefly review each and choose the one that best suits our needs.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">Autoregressive models</span>
</div>


These models treat text-to-image generation as a sequence generation task. A decoder-only Transformer takes a sequence of text tokens as input and outputs a sequence of visual tokens representing an image. An image tokenizer then decodes these visual tokens into the actual image.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-3-EVPPXBGI.svg" alt="Figure 3: Autoregressive text-to-image generation" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 3: Autoregressive text-to-image generation</p>
</div>


Several text-to-image models have been developed using this approach, such as OpenAI’s DALL-E [5] and Google’s Muse [6].


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">Diffusion models</span>
</div>


First introduced in 2019 [7], diffusion models gained mainstream attention about three years later. They use a different approach for text-to-image generation by starting with random noise and gradually transforming it into a clear image based on the text prompt. This process typically involves a text encoder, such as OpenAI’s CLIP [8] or Google’s T5 [9], which converts the text prompt into an embedding. This embedding captures the meaning of the prompt and guides the diffusion model to generate images that match it.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-4-CZWRHW7T.svg" alt="Figure 4: Diffusion-based text-to-image generation" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 4: Diffusion-based text-to-image generation</p>
</div>


Examples of diffusion-based text-to-image models include Google’s Imagen 3 [2], OpenAI’s DALL-E 2 [10], and Stability AI’s Stable Diffusion [11].


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">Diffusion versus autoregressive models</span>
</div>


Autoregressive models frame text-to-image generation as a sequence generation task, while diffusion models approach it as an iterative refinement process. This key difference in modeling impacts their capabilities.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-5-2SDEBR35.svg" alt="Figure 5: Diffusion vs. autoregressive model characteristics" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 5: Diffusion vs. autoregressive model characteristics</p>
</div>


Both diffusion and autoregressive models can produce realistic images, are slow in generation, typically have billions of parameters, and require substantial computational resources for training. Despite these similarities, they differ in three key aspects:


* 1. Implementation complexity: Autoregressive models are simpler to implement during both training and inference. During training, they are more statistically efficient because they can obtain useful gradient signals from all steps in a single forward-backward pass. In contrast, diffusion models are less statistically efficient, requiring sampling of different noise levels for each training example. At inference time, once the Transformer in an autoregressive model generates the sequence of visual tokens, these tokens form the final image. Diffusion models, however, refine the image over many steps, adding complexity to the implementation.
* 2. Image quality: Diffusion models have shown better performance in generating highly detailed and realistic images. Their iterative process allows the model to continuously refine and enhance fine details, leading to superior overall realism in the generated images.
* 3. Flexibility in sampling: Diffusion models are more flexible in trading off sampling speed and image quality. They can easily adjust the number of sampling steps—more steps usually lead to higher-quality images but take more time. Once trained, an autoregressive model cannot easily make such adjustments.


In this chapter, we choose diffusion models to prioritize exceptional image quality. In the model development section, we explore the architecture, training methods, and sampling techniques of diffusion models.


<div style="border-left: 5px solid #d90b5c; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(217,11,92,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Data Preparation</span>
</div>


Our dataset consists of approximately 500 million image-caption pairs. However, large-scale datasets often require substantial preprocessing before they can be used for model training. In this section, we explore common techniques to prepare images and captions for diffusion training.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">Image preparation</span>
</div>


We focus on two main steps to prepare the images: filtering inappropriate images and standardizing the remaining ones. Let's delve into each step in more detail.


<strong style="color: #f80e5e;">Filtering inappropriate images</strong>


In large-scale datasets, many images may not be useful for training. It's crucial to remove these to ensure the model learns only from high-quality and safe data. To achieve this, we perform the following steps:


* Remove small images: We discard image-caption pairs where the image size is smaller than a certain threshold, for example, 64x64 pixels. Small images are often of poor quality and may not provide valuable information for training.
* Deduplicate images: We use deduplication methods, such as [12], to remove identical or perceptually similar images. This prevents the model from being biased toward certain images that appear more frequently.
* Remove inappropriate images: We employ harm detection and NSFW (Not Safe For Work) detection models to filter out harmful content such as violence or nudity. This ensures the model doesn't learn to generate inappropriate images.
* Remove low-aesthetic images: We use specialized ML models to eliminate images that aren't aesthetically pleasing. This helps the model focus on high-quality images during training.


<strong style="color: #f80e5e;">Standardizing images</strong>


* Adjust image dimensions: The diffusion model requires inputs of specific dimensions. Therefore, it's crucial to have training data of similar sizes. For example, if the expected model input is 128x128, we first resize the images so that the smaller dimension is 128, preserving the aspect ratio. Then, we center-crop them to achieve the final size of 128x128.
* Normalize images: We normalize pixel values to a standard range such as [0, 1] or [-1, 1] for more stable training.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-6-ZMT5EU2U.svg" alt="Figure 6: Image preparation steps" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 6: Image preparation steps</p>
</div>


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">Caption preparation</span>
</div>


In addition to images, captions are often irrelevant or missing. Here are common steps to ensure captions are consistent and of high quality:


* Handle missing or non-English captions: For images without captions or with captions in another language, we use an image captioning model such as BLIP-3 [13] to automatically generate descriptive captions. If you would like to build an image captioning system from scratch, refer to Chapter 5.
* Enhance captions: We use a pretrained model such as CLIP [8] to score the relevance of each image-caption pair. For pairs scoring below a threshold, we replace the original caption with an auto-generated one using the BLIP-3 model.
* Remove poorly matched pairs: After enhancing captions, we remove image-caption pairs with CLIP similarity scores below a threshold. This step ensures that the model is exposed only to pairs whose captions accurately describe the images.


<div style="border-left: 5px solid #d90b5c; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(217,11,92,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Model Development</span>
</div>


<div style="border-bottom: 2px solid rgba(248,14,94,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #f80e5e;">Architecture</span>
</div>


A diffusion model, as previously explained, progressively denoises a noisy image through multiple steps until it becomes clear. In each step, as illustrated in Figure 7, the model takes a noisy image as input and predicts the noise to be removed. Two common architectures are typically used for this purpose:


* U-Net
* Diffusion Transformer (DiT)


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-7-M7MESQJ3.svg" alt="Figure 7: Input and output of the diffusion model in a single step" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 7: Input and output of the diffusion model in a single step</p>
</div>


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">U-Net</span>
</div>


U-Net [14] is a convolutional neural network (CNN) architecture originally developed for biomedical image segmentation. It consists of a series of downsampling blocks followed by upsampling blocks, as shown in Figure 8.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-8-3DCGJCBD.svg" alt="Figure 8: U-Net downsampling and upsampling blocks" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 8: U-Net downsampling and upsampling blocks</p>
</div>


<strong style="color: #f80e5e;">Downsampling blocks</strong>


The downsampling blocks progressively reduce spatial dimensions (height and width) while increasing depth (number of channels), leading to a compressed representation of the input. Each downsampling block typically consists of the following:


* Convolution operation: Extracts visual features from the input.
* Batch normalization: Normalizes feature maps to stabilize training.
* Non-linear activation: Introduces non-linearity to learn complex patterns.
* Max-pooling: Reduces the feature map dimensions.
* Cross-attention: Cross-attends to additional conditions, such as text prompt tokens. This is necessary to ensure the text prompt influences the predicted noise.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-9-UG7YRW2U.svg" alt="Figure 9: Typical layers in a downsampling block" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 9: Typical layers in a downsampling block</p>
</div>


Let's explore the cross-attention layer in more detail, as this is the first instance where we're applying cross-attention between different modalities-image and text inputs. The text prompt is processed by a text encoder, such as a Transformer-based model, which converts words or tokens into a sequence of continuous embeddings. These embeddings capture the semantic meaning of the text. During each denoising step of the diffusion process, the model receives a noisy image as input and processes it through layers like Conv2D and BatchNorm2D to extract visual features. In the cross-attention layer, the queries are derived from the image features, while the keys and values come from the text embeddings. This enables the model to align and integrate information effectively from the text into the image features.


<strong style="color: #f80e5e;">Upsampling blocks</strong>


The upsampling blocks symmetrically increase spatial dimensions and decrease feature map depth. The final output matches the original input size, which, in this case, is the predicted noise. Each upsampling block consists of the following:


* Transposed convolution: Uses operations like PyTorch’s ConvTranspose2D to process and increase the feature map’s dimensions.
* Batch normalization: Normalizes feature maps to stabilize training.
* Non-linear activation: Introduces non-linearity to learn complex patterns.
* Cross-attention: Maintains the influence of additional conditions during upsampling.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-10-5P7MJQ2J.svg" alt="Figure 10: Typical layers in an upsampling block" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 10: Typical layers in an upsampling block</p>
</div>


The U-Net architecture has various details and variations. Different implementations may use different layers and configurations. However, understanding the key components and structure is generally sufficient for most ML system design interviews. For more in-depth information, refer to [14].


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">DiT</span>
</div>


DiT [15] is another popular architecture in diffusion models. Unlike U-Net, which uses a series of downsampling and upsampling layers, DiT primarily relies on a Transformer architecture to process the noisy input image and predict the noise.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-11-T77AHJII.svg" alt="Figure 11: DiT components" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 11: DiT components</p>
</div>


DiT is primarily inspired by the Vision Transformer (ViT) [16] architecture discussed in Chapter 5. DiT components are:


* Patchify: Converts the input image into a sequence of patch embeddings.
* Positional encoding: Attaches position information to each patch embedding to indicate its location in the original image.
* Transformer: Processes the sequence of embeddings and other conditioning signals, such as the text prompt, to predict the noise for each patch.
* Unpatchify: Converts the sequence of predicted noise vectors into an image with the same dimension as the original input image.


In summary, both U-Net and DiT architectures perform well in practice. U-Net was originally used in several text-to-image models such as Google’s Imagen [17] and Stability AI’s Stable Diffusion [11]. Recently, the DiT architecture has shown great promise for text-to-image generation. For educational purposes, we use the U-Net architecture in this chapter. Chapter 11 explores the DiT architecture in more detail.


<div style="border-bottom: 2px solid rgba(248,14,94,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #f80e5e;">Training</span>
</div>


A diffusion model is trained by employing the diffusion process. The diffusion process has two phases:


* Forward process
* Backward process


<strong style="color: #f80e5e;">Forward process</strong>


In the forward process, also known as the noising process, noise is gradually added to an image over multiple steps (denoted as $t$ or timesteps) until the image becomes completely noisy. The value of $t$, representing the number of steps, is typically chosen randomly from a range, usually between 1 and 1,000. The forward process does not involve any ML models or parameter updates.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-12-I7PC7S65.svg" alt="Figure 12: Forward diffusion process" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 12: Forward diffusion process</p>
</div>


<strong style="color: #f80e5e;">Backward process</strong>


In the backward process, also known as the denoising process, an ML model learns to reverse the forward process. At each step, the model predicts the noise in the noisy image. This predicted noise is then used to reduce the noise in the input image. As shown in Figure 13, this process is repeated until the image becomes clear.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-13-ZFWFVMDK.svg" alt="Figure 13: Backward diffusion process" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 13: Backward diffusion process</p>
</div>


With an understanding of both the forward and backward processes, we can now explore how they are applied during the diffusion training process.


<strong style="color: #f80e5e;">Diffusion training process</strong>


During training, we introduce noise to the original image by simulating the forward process, then ask the model to predict this noise. This process involves four key steps:


* 1. Noise addition
* 2. Preparation of conditioning signals
* 3. Noise prediction
* 4. ML objective and loss calculation


This section includes some math for those who are interested, but the details do not affect the design of the ML system.


<strong style="color: #f80e5e;">1. Noise addition</strong>


The first step is to simulate the forward diffusion process by adding noise to the original image over multiple timesteps. At each timestep, we corrupt the image slightly by adding Gaussian noise. This gradual addition of noise transforms the image into pure noise over time.

The amount of noise added at each timestep is controlled by a noise schedule. The noise schedule is defined by a set of variance parameters, $\beta_1, \beta_2, \dots, \beta_T$, where $T$ is the total number of timesteps. Each $\beta_t \in (0,1)$ controls the amount of noise added at timestep $t$.

The noise schedule typically increases the values of $\beta$ incrementally:


$$ \beta_1 < \beta_2 < \ldots < \beta_T $$


Thus, smaller amounts of noise are added in the early steps, preserving more of the original image, and larger amounts of noise are added in the later steps, accelerating the diffusion process.

With the noise schedule defined, we can express the noisy data at timestep $t$ using the noise addition formula:


$$ x_t = \sqrt{1-\beta_t}x_{t-1} + \sqrt{\beta_t}\epsilon $$


where:


* $x_t$ is the noisy image at timestep $t$,
* $x_{t-1}$ is the noisy image at timestep $t-1$,
* $\epsilon$ is the Gaussian noise sampled from the standard normal distribution $\mathcal{N}(0, I)$,
* $\beta_t$ is the variance schedule parameter at timestep $t$, controlling the amount of noise added.


Iteratively adding noise over many steps can be time-consuming. Instead, it can be shown that the noisy data at timestep $t$ can be directly derived from the original data, $x_0$:


$$ \begin{aligned} & x_t=\sqrt{\alpha_t} x_{t-1}+\sqrt{1-\alpha_t} \epsilon_{t-1} \\ & =\sqrt{\alpha_t}\left(\sqrt{\alpha_{t-1}} x_{t-2}+\sqrt{1-\alpha_{t-1}} \epsilon_{t-2}\right)+\sqrt{1-\alpha_t} \epsilon_{t-1} \\ & =\sqrt{\alpha_t \alpha_{t-1}} x_{t-2}+\sqrt{1-\alpha_t \alpha_{t-1}} \epsilon_{t-2}^{\prime} \\ & =\ldots \\ & =\sqrt{\alpha_t^{\prime}} x_0+\sqrt{1-\alpha_t^{\prime}} \epsilon \end{aligned} $$


where:


* $x_t$ is the noisy image at timestep $t$,
* $\alpha_t=1-\beta_t$ and $\alpha_t^{\prime}=\prod_{i=1}^t \alpha_i$ are reparameterizations of $\beta_t$,
* $\epsilon$ is the Gaussian noise sampled from the standard normal distribution $\mathcal{N}(0, I)$.


In summary, during noise addition, we randomly sample $t$ and compute $x_t$ directly from $x_0$, without the need to iteratively add noise at each timestep, using the following formula:


$$ x_t = \sqrt{\alpha_t^{\prime}} x_0+\sqrt{1-\alpha_t^{\prime}} \epsilon $$


<strong style="color: #f80e5e;">2. Preparation of conditioning signals</strong>


To predict the added noise, the model typically expects two additional pieces of information: the image caption and the sampled timestep, $t$, which indicates the noise level. We use separate encoders (see Figure 14) to prepare each of these conditioning signals to be processed by the model.


<strong style="color: #f80e5e;">3. Noise prediction</strong>


The primary goal of training a diffusion model is to learn how to reverse the forward diffusion process-that is, to reconstruct the original data, $x_0$, from its noisy version, $x_t$. It has been shown that directly predicting $x_0$ is not as effective. Instead, training the model to predict the noise, $\epsilon$, added during the forward process simplifies the task and improves performance. Therefore, in this step, the model predicts the noise, $\epsilon$, given the noisy input, $x_t$, and the timestep, $t$.


<strong style="color: #f80e5e;">4. ML objective and loss calculation</strong>


The ML objective is to minimize the difference between the true noise, $\epsilon$, and the model's prediction. The loss function used is the mean squared error (MSE) between the true noise and the predicted noise:


$$ L=E_{t, x_0, \epsilon}\left(\left\|\epsilon-\epsilon_\theta\left(x_t, t\right)\right\|^2\right) $$


where:


* $t$ is a timestep sampled uniformly from $\{1,2,\ldots,T\}$,
* $\epsilon \sim \mathcal{N}(0, I)$ is the Gaussian noise used in the forward process,
* $x_t$ is the noisy data at timestep $t$, computed using the noise addition formula,
* $\epsilon_\theta(x_t, t)$ is the prediction of the neural network model (U-Net or DiT).


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-14-RGHHCRW3.svg" alt="Figure 14: A single iteration in diffusion training" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 14: A single iteration in diffusion training</p>
</div>


To enhance the reading experience, we have omitted the mathematical details, such as the derivation of the tractable mean and the loss simplification. For more information on diffusion training, refer to [18].


<div style="border-bottom: 2px solid rgba(248,14,94,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #f80e5e;">Sampling</span>
</div>


Sampling refers to generating a new image from a trained diffusion model. In this section, we’ll explore how sampling works in diffusion models and how noises are transformed into coherent images guided by the text prompt.

The sampling process starts with an image of random pixels, typically drawn from a Gaussian distribution. The model then gradually refines this image step by step. At each step, the diffusion model predicts the noise present in the current image and uses this prediction to adjust the image slightly in the right direction. This gradual refinement continues, each step producing a clearer image until a clear and detailed image is achieved.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-15-2AGKPRGO.svg" alt="Figure 15: Sampling process" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 15: Sampling process</p>
</div>


The basic sampling process described above has two drawbacks. First, it often fails to generate images that accurately match the text prompt. Second, it is slow, because generating each sample requires many iterative steps. The following two techniques are commonly employed in practice to mitigate the issues described above:


* Classifier-free guidance (CFG): CFG [19] improves the alignment between images and text prompts in diffusion models. During training, the model learns to generate images with and without the text prompt. During sampling, CFG adjusts the balance between these two modes. CFG ensures the generated images closely match the text prompts by increasing the influence of the conditioned (text prompt) mode and reducing the unconditioned (no text prompt) mode. This adjustment guides the diffusion process to produce more accurate results. To learn more about CFG, refer to [19].
* Reduction of diffusion steps: Sampling algorithms such as DDIM [20] reduce the number of diffusion steps from the standard 1,000 to as few as 20. This significantly speeds up the generation process while maintaining image quality. To learn more about DDIM, refer to [20].


In most ML system design interviews, the focus is on high-level concepts and how components work together, not on the intricate details. If you are interested in exploring diffusion models in more depth, refer to [21][19][20].


<div style="border-bottom: 2px solid rgba(248,14,94,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #f80e5e;">Challenges in text-to-image diffusion models</span>
</div>


Diffusion models are typically very large. For example, DALLE-2 has 3.5 billion parameters [10]. This capacity is necessary since these models have to learn diverse concepts, shapes, and styles.

Training such large models poses several challenges during both training and sampling. The most common ones include:


* Resource-intensive model training
* Slow image generation


<strong style="color: #f80e5e;">Resource-intensive model training</strong>


Training diffusion models is computationally intensive, requiring significant processing power. It also requires substantial GPU memory due to the size of the model and the high-dimensional nature of the generated images. Most modern GPUs may not have sufficient memory to fit model parameters, activations, and gradients during training. The following strategies are commonly employed to overcome these challenges:


* Mixed precision training: This technique uses both 16-bit and 32-bit floating-point types to reduce memory usage and increase computational efficiency. To learn more, refer to [22].
* Model and data parallelism: These methods distribute training across multiple devices. Most distributed training frameworks, such as FSDP [23] and Deepspeed [24], support various parallelism techniques.
* Latent diffusion models: These models operate in a lower-dimensional space instead of pixel space, significantly speeding up training and inference. Chapter 11 examines this approach in more detail.


<strong style="color: #f80e5e;">Slow image generation</strong>


Generating images from text in diffusion models is slow for two main reasons. First, due to the sequential nature of the diffusion sampling process, multiple steps are needed to refine the image. Second, as diffusion models have billions of parameters, significant computations occur at each step.

Common strategies to mitigate this challenge include:


* Parallel sampling: Implementing parallel processing during sampling reduces the time needed to generate images [25].
* Model distillation: A distilled model improves generation speed because of its reduced size, but it retains the behavior and performance of the original model. To learn more about model distillation in diffusion models, refer to [26].
* Model quantization: This technique reduces the precision of the model's weights, which decreases memory usage and speeds up generation.


<div style="border-left: 5px solid #d90b5c; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(217,11,92,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Evaluation</span>
</div>


<div style="border-bottom: 2px solid rgba(248,14,94,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #f80e5e;">Offline evaluation metrics</span>
</div>


A consistent and reliable benchmark is key to evaluating text-to-image models. DrawBench [17] serves this purpose by providing a curated set of prompts that test various aspects of image generation such as object composition, interaction, and context understanding. These prompts, ranging from simple to complex, help to assess how accurately the model generates images from text. Due to its comprehensiveness, we use DrawBench to evaluate our text-to-image model. Let’s examine both automated metrics and human evaluation to assess three key areas of the model’s ability to generate images:


* Image quality
* Image diversity
* Image-text alignment


As discussed in previous chapters, Inception score (IS) [27] and Fréchet Inception distance (FID) [28] are two common metrics for evaluating quality and diversity in image generation systems. In this section, we focus primarily on image-text alignment.


<strong style="color: #f80e5e;">Image-text alignment</strong>


Image-text alignment refers to how accurately the generated images match the text prompts. Measuring this alignment is important because it ensures the generated images are faithful to the user's input. A common metric for assessing this is CLIPScore [29], which evaluates the degree of alignment. Before diving into CLIPScore, let's briefly review CLIP.


<strong style="color: #f80e5e;">CLIP</strong>


CLIP [8] is a model developed by OpenAI that has been trained to match images with their corresponding descriptions. It consists of two encoders: one for text and one for images. The text encoder converts input text into a text embedding; the image encoder converts the image into an image embedding.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-16-FBA7BGBF.svg" alt="Figure 16: CLIP encoders" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 16: CLIP encoders</p>
</div>


During training, CLIP learns to align embeddings by bringing related text and image embeddings closer together and pushing unrelated ones apart. This helps CLIP develop a shared embedding space where both an image and its associated text will map into the same space.

After training, similar text descriptions map close to each other in the embedding space, and images map near their relevant descriptions.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-17-BXSJL66Y.svg" alt="Figure 17: Text and image feature vectors mapped to the CLIP embedding space" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 17: Text and image feature vectors mapped to the CLIP embedding space</p>
</div>


With an understanding of the CLIP model, we can now easily explore CLIPScore as a metric for evaluating image-text alignment.


<strong style="color: #f80e5e;">CLIPScore</strong>


The CLIPScore measures the cosine similarity between the CLIP embeddings of a text description and an image. It shows how closely the image matches the text in the high-dimensional embedding space. A higher score indicates better alignment between an image and a description.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">Human evaluation</span>
</div>


Human evaluation complements the automated metrics by assessing both image quality and text alignment using the following approach:


* Image quality: Human raters compare a generated image with a reference image by judging which of the images is more photorealistic. Image quality is determined by the percentage of times the generated image is chosen over the reference image.
* Text alignment: Human raters are shown an image and its caption and then asked, "Does the caption accurately describe the above image?" The responses "yes," "somewhat," and "no" are scored as 100, 50, and 0, respectively. These scores are averaged separately for generated images and reference images to measure text alignment.


<div style="border-bottom: 2px solid rgba(248,14,94,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #f80e5e;">Online evaluation metrics</span>
</div>


Online metrics measure how the model works in production. Common metrics to evaluate our text-to-image model include:


* Click-through rate (CTR): The percentage of users who click on the generated images. A high CTR indicates that users find the generated images useful.
* Time spent on page: The average time users spend with the service. Longer viewing times indicate higher user engagement.
* User feedback: Direct feedback from users is collected through feedback. Positive feedback indicates satisfaction with the image quality and text alignment.
* Conversion rate: The percentage of users who take a desired action (e.g., purchase, sign up) after interacting with generated images. A high conversion rate indicates satisfaction with the model's performance.
* Latency: The time it takes to generate an image from a text prompt. Lower latency indicates faster performance, which is important for user satisfaction.
* Throughput: The number of image generations the model can handle per second. High throughput ensures the service can serve more users.
* Resource utilization: The computational resources (e.g., CPU, GPU, memory) used to run the model and serve users. Efficient resource utilization is key to reducing costs.
* Average cost per user per month: Generating images with models that have billions of parameters is costly. If users are unhappy with the images, they might repeatedly generate new ones using the same prompt but different seeds, hoping for better results. This behavior increases our expenses. By monitoring this metric, we can ensure that costs remain justifiable.


<div style="border-left: 5px solid #d90b5c; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(217,11,92,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Overall ML System Design</span>
</div>


While the diffusion model is at the core of a text-to-image generation system, several other pipelines are crucial for ensuring efficiency, safety, and quality. In this section, we delve into the holistic design of a text-to-image generation system by examining the following pipelines:


* Data pipeline
* Training pipeline
* Model optimization pipeline
* Inference pipeline


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">Data pipeline</span>
</div>


The data pipeline prepares data for training by removing inappropriate images, standardizing the rest, and storing them. It ensures captions are present and relevant and uses a pretrained model such as Google’s T5 [9] to pre-compute and cache caption embeddings. This caching reduces computation during training.

In addition to preparing text-image pairs from the training data, the pipeline also collects and processes newly generated data such as user prompts, generated images, and user feedback. This new data is added to the training set for future use.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-18-SKS5QNJI.svg" alt="Figure 18: Data pipeline" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 18: Data pipeline</p>
</div>


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">Training pipeline</span>
</div>


The training pipeline trains a model using the latest training data collected by the data pipeline.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-19-KFI7RXYV.svg" alt="Figure 19: Training pipeline" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 19: Training pipeline</p>
</div>


The training pipeline ensures that the model adapts to recent user prompts and is trained on higher-quality generated images.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">Evaluation pipeline</span>
</div>


The evaluation pipeline assesses newly trained models using predefined automated metrics to determine whether or not they meet performance and quality standards for deployment.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">Model optimization pipeline</span>
</div>


The model optimization pipeline is responsible for enhancing the efficiency of the model. There are several methods to optimize models:


* Model compression: Use techniques such as quantization and pruning to reduce model size and generation time.
* Model distillation: Distill the model into a smaller one to reduce the model size and generation time.
* Optimized algorithms: Replace sampling with more efficient algorithms for faster generation.


Once the model optimization is completed, the optimized model can replace the existing model in production.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #f80e5e; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #f80e5e;">Inference pipeline</span>
</div>


The inference pipeline handles user requests and generates images based on text prompts. It comprises several components, each playing an important role in ensuring the system's quality and safety. In this section, we will examine the key components:


* Prompt auto-complete
* Prompt safety service
* Prompt enhancement
* Image generation
* Harm detection
* Super-resolution service


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-20-MCBVLGYN.svg" alt="Figure 20: Inference pipeline" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 20: Inference pipeline</p>
</div>


<strong style="color: #f80e5e;">Prompt auto-complete service</strong>


The prompt auto-complete service uses a specialized model to suggest possible next words or phrases in real-time as the user types their prompt. This enhances user experience by providing potential completions.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-21-W236GIPQ.svg" alt="Figure 21: Suggested phrases by the auto-complete service" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 21: Suggested phrases by the auto-complete service</p>
</div>


<strong style="color: #f80e5e;">Prompt safety service</strong>


This service uses a text classification model to process user prompts and reject those that violate our usage policies, for example, requests for violence, hateful imagery, or nudity.

This service ensures that the system adheres to safety standards and prevents the generation of inappropriate images.


<strong style="color: #f80e5e;">Prompt enhancement</strong>


The prompt enhancement component refines user prompts to improve their clarity, coherence, and details.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-22-G65RYMT6.svg" alt="Figure 22: An example of prompt enhancement" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 22: An example of prompt enhancement</p>
</div>


This component is widely used in advanced image and video generation systems [30], as it effectively helps the model produce better outputs. It improves the quality of generated images by offering the model a more coherent and detailed prompt.


<strong style="color: #f80e5e;">Image generation</strong>


The image generation component is the core of the inference pipeline. It interacts with the T5 text encoder to encode the enhanced text prompt into a sequence of tokens. These tokens are passed to the diffusion model to generate one or multiple images for each prompt.


<strong style="color: #f80e5e;">Harm detection</strong>


This component ensures that generated images are safe for users. If an image still contains violence or nudity despite previous safeguards, the component flags it and blocks it from being shown.


<strong style="color: #f80e5e;">Super-resolution service</strong>


The super-resolution service increases the resolution of the generated images. This step ensures that the final output is visually appealing and meets resolution requirements.

In practice, text-to-image systems often use at least one super-resolution model because diffusion models typically cannot generate high-resolution images directly. Instead, the diffusion model is trained at a lower resolution, and specialized super-resolution models enhance the resolution. For example, the base model might generate a 64x64 image, which a first super-resolution model increases to 256x256, followed by a second model that boosts it to 1024x1024. Google's [31] follows this approach to achieve the desired resolution.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-9-23-YHAFPZV3.svg" alt="Figure 23: A cascade of super-resolution models" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 23: A cascade of super-resolution models</p>
</div>


In summary, various pipelines work together to ensure that a text-to-image system is reliable, high-quality, and safe. The data pipeline provides the foundation for continuous improvement, while the training and model optimization pipelines enhance the model's performance. The inference pipeline ensures safe, efficient, and high-quality image generation. These pipelines create a system ready for real-world challenges.


<div style="border-left: 5px solid #d90b5c; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(217,11,92,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Other Talking Points</span>
</div>


If time permits at the end of the interview, consider discussing these additional topics:


* Using consistency models for faster image generation [26].
* Employing RLHF for quality improvement [32].
* Extending the text-to-image model to support inpainting and outpainting applications [33].
* Personalizing the text-to-image model to a particular concept (Chapter 10).
* Details of different scheduling techniques [34].
* Details of DDPM and DDIM, including their theoretical foundations [20][18].
* Supporting multiple aspect ratios and resolutions using techniques such as Patch n’ Pack [35].
* Details of developing a re-captioning model [36][37][13].
* Improving the diversity-fidelity trade-off with guidance [19].
* More advanced controls over the generated images using techniques like ControlNet [38].
* Controlling the style of the generated images [39].


<div style="border-left: 5px solid #d90b5c; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(217,11,92,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Reference Material</span>
</div>


[1] OpenAI’s DALL-E 3. https://openai.com/index/dall-e-3/.

[2] Imagen 3. https://arxiv.org/abs/2408.07009.

[3] Adobe’s Firefly. https://www.adobe.com/products/firefly.html.

[4] Introducing ChatGPT. https://openai.com/index/chatgpt/.

[5] Zero-Shot Text-to-Image Generation. https://arxiv.org/abs/2102.12092.

[6] Muse: Text-To-Image Generation via Masked Generative Transformers. https://arxiv.org/abs/2301.00704.

[7] Generative Modeling by Estimating Gradients of the Data Distribution. https://arxiv.org/abs/1907.05600.

[8] Learning Transferable Visual Models From Natural Language Supervision. https://arxiv.org/abs/2103.00020.

[9] Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer. https://arxiv.org/abs/1910.10683.

[10] Hierarchical Text-Conditional Image Generation with CLIP Latents. https://arxiv.org/abs/2204.06125.

[11] High-Resolution Image Synthesis with Latent Diffusion Models. https://arxiv.org/abs/2112.10752.

[12] On the De-duplication of LAION-2B. https://arxiv.org/abs/2303.12733.

[13] xGen-MM (BLIP-3): A Family of Open Large Multimodal Models. https://www.arxiv.org/abs/2408.08872.

[14] U-Net: Convolutional Networks for Biomedical Image Segmentation. https://arxiv.org/abs/1505.04597.

[15] Scalable Diffusion Models with Transformers. https://arxiv.org/abs/2212.09748.

[16] An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale. https://arxiv.org/abs/2010.11929.

[17] Photorealistic Text-to-Image Diffusion Models with Deep Language Understanding. https://arxiv.org/abs/2205.11487.

[18] Denoising Diffusion Probabilistic Models. https://arxiv.org/abs/2006.11239.

[19] Classifier-Free Diffusion Guidance. https://arxiv.org/abs/2207.12598.

[20] Denoising Diffusion Implicit Models. https://arxiv.org/abs/2010.02502.

[21] Introduction to Diffusion Models. https://lilianweng.github.io/posts/2021-07-11-diffusion-models/.

[22] Mixed Precision Training. https://arxiv.org/abs/1710.03740.

[23] FSDP tutorial. https://pytorch.org/tutorials/intermediate/FSDP_tutorial.html.

[24] DeepSpeed. https://github.com/microsoft/DeepSpeed.

[25] Parallel Sampling of Diffusion Models. https://arxiv.org/abs/2305.16317.

[26] Consistency Models. https://arxiv.org/abs/2303.01469.

[27] Inception score. https://en.wikipedia.org/wiki/Inception_score.

[28] FID calculation. https://en.wikipedia.org/wiki/Fr%C3%A9chet_inception_distance.

[29] CLIPScore: A Reference-free Evaluation Metric for Image Captioning. https://arxiv.org/abs/2104.08718.

[30] Sora overview. https://openai.com/index/video-generation-models-as-world-simulators/.

[31] Imagen Video: High Definition Video Generation with Diffusion Models. https://arxiv.org/abs/2210.02303.

[32] Finetune Stable Diffusion Models with DDPO via TRL. https://huggingface.co/blog/trl-ddpo.

[33] Kandinsky: an Improved Text-to-Image Synthesis with Image Prior and Latent Diffusion. https://arxiv.org/abs/2310.03502.

[34] On the Importance of Noise Scheduling for Diffusion Models. https://arxiv.org/abs/2301.10972.

[35] Patch n' Pack: NaViT, a Vision Transformer for any Aspect Ratio and Resolution. https://arxiv.org/abs/2307.06304.

[36] InternVL: Scaling up Vision Foundation Models and Aligning for Generic Visual-Linguistic Tasks. https://arxiv.org/abs/2312.14238.

[37] BLIP-2: Bootstrapping Language-Image Pre-training with Frozen Image Encoders and Large Language Models. https://arxiv.org/abs/2301.12597.

[38] Adding Conditional Control to Text-to-Image Diffusion Models. https://arxiv.org/abs/2302.05543.

[39] StyleDrop: Text-to-image generation in any style. https://research.google/blog/styledrop-text-to-image-generation-in-any-style/.
